In [1]:
from lcdb.db import LCDB
from tqdm import tqdm
import pandas as pd

In [2]:
workflow_mapping = {
  "libsvm": "lcdb.workflow.sklearn.LibSVMWorkflow",
  "randomforest": "lcdb.workflow.sklearn.RandomForestWorkflow",
  "knn": "lcdb.workflow.sklearn.KNNWorkflow",
  "xgboost": "lcdb.workflow.xgboost.XGBoostWorkflow",
  "treesensemble": "lcdb.workflow.sklearn.TreesEnsembleWorkflow",
  "liblinear": "lcdb.workflow.sklearn.LibLinearWorkflow"
}

In [3]:
workflow = "liblinear"
workflow_class = workflow_mapping[workflow]

In [4]:
import pandas as pd

# Read CSV file (no header, single column)
df = pd.read_csv('experiments/surf/snellius/datasets_to_test.csv', header=None)

# Extract OpenML IDs
openmlids = df.iloc[:, 0].tolist()

# Print the first few values
print("Raw OpenML IDs from CSV:", openmlids[:10])
print("Type of values in list:", [type(x) for x in openmlids[:10]])


Raw OpenML IDs from CSV: [3, 12, 23, 31, 54, 181, 188, 1049, 1067, 1111]
Type of values in list: [<class 'int'>, <class 'int'>, <class 'int'>, <class 'int'>, <class 'int'>, <class 'int'>, <class 'int'>, <class 'int'>, <class 'int'>, <class 'int'>]


In [ ]:
import numpy as np
openmlids = np.array(openmlids, dtype=int).tolist()

In [ ]:
lcdb = LCDB()
stats_info = lcdb.statistics(
    openmlids=[188, 12, 42769],
    # openmlids=openmlids,
    workflows=[workflow_class],
    campaigns=['data_probing-32'],
    validation_seeds=[0],
    test_seeds=[0],
    show_progress=True
)

# print("Non-duplicate Error: \n", debug_info['errors'].drop_duplicates().reset_index(drop=True).to_frame().to_string(header=False))
df = stats_info

Skipping non-existent repository: <lcdb.db._local_repository.LocalRepository object at 0x1483d42cce50>


100%|██████████| 3/3 [00:02<00:00,  1.02it/s]


In [8]:
# for size in [10, 20, 30, 40, 50, len(openmlids)]:
#     print(f"\nTrying with {size} IDs...")

#     try:
#         debug_info = lcdb.statistics(
#             openmlids=openmlids[:size],  # Use only a subset
#             workflows=[workflow_class],
#             campaigns=['data_probing-32'],
#             validation_seeds=[0],
#             test_seeds=[0],
#             show_progress=True
#         )
#         print(f"Success with {size} IDs")
#     except Exception as e:
#         print(f"Failed at {size} IDs. Error: {e}")
#         break  # Stop at failure point


In [9]:
df

,workflow,openmlid,num_configs,error_rate,tracebacks,errors,memory,avg_config_time,max_config_time
0,lcdb.workflow.sklearn.LibLinearWorkflow,188,128,0.015625,"[""Traceback (most recent call last):\n File ""...",[ValueError: zero-size array to reduction oper...,233111552,4.594236,78.482174
1,lcdb.workflow.sklearn.LibLinearWorkflow,12,128,0.078125,"[""Traceback (most recent call last):\n File ""...",[FunctionCallTimeoutError: Function timeout ex...,266407936,155.616310,1782.776965
2,lcdb.workflow.sklearn.LibLinearWorkflow,42769,128,0.117188,"[""Traceback (most recent call last):\n File ""...",[FunctionCallTimeoutError: Function timeout ex...,1129689088,509.805380,1684.452181


In [ ]:
import pandas as pd
import numpy as np

# memory from bytes to GB
df["memory_gb"] = df["memory"] / (1024**3)

memory_bins = [0, 1, 2, 4, 8, 16, 24, 32, 40, float("inf")]
memory_labels = ["<1GB", "1GB", "2GB", "4GB", "8GB", "16GB", "24GB", "32GB", "40GB+"]

# memory bins assigned
df["memory_bin"] = pd.cut(df["memory_gb"], bins=memory_bins, labels=memory_labels, right=False)

# group by workflow and memory_bin, listing datasets per bin
grouped_workflow = df.groupby(["workflow", "memory_bin"])["openmlid"].unique().reset_index()

grouped_workflow["openmlid"] = grouped_workflow["openmlid"].apply(lambda x: list(x) if isinstance(x, (np.ndarray, list)) else [])

# remove rows with no openmlids
grouped_workflow = grouped_workflow[grouped_workflow["openmlid"].map(len) > 0]

grouped_workflow


/scratch-local/67862/ipykernel_2840693/2870303608.py:15: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  grouped_workflow = df.groupby(["workflow", "memory_bin"])["openmlid"].unique().reset_index()


,workflow,memory_bin,openmlid
0,lcdb.workflow.sklearn.LibLinearWorkflow,<1GB,"[188, 12]"
1,lcdb.workflow.sklearn.LibLinearWorkflow,1GB,[42769]


In [ ]:
import pandas as pd

# memory from bytes to GB
df["memory_gb"] = df["memory"] / (1024**3)  

memory_bins = [0, 1, 2, 4, 8, 16, 24, 32, 40, float("inf")]
memory_labels = ["<1GB", "1GB", "2GB", "4GB", "8GB", "16GB", "24GB", "32GB", "40GB+"]

# each row is assigned to memory bin
df["memory_bin"] = pd.cut(df["memory_gb"], bins=memory_bins, labels=memory_labels, right=False)

# grouped by memory bin and collect (workflow, openmlid) pairs as a list
grouped = df.groupby("memory_bin").agg(
    workflow_dataset_combinations=("workflow", lambda x: list(set(zip(x, df.loc[x.index, "openmlid"]))))
).reset_index()

# remove bins with no tuples
grouped = grouped[grouped["workflow_dataset_combinations"].map(len) > 0]

grouped


/scratch-local/67862/ipykernel_2840693/1195834284.py:14: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  grouped = df.groupby("memory_bin").agg(


,memory_bin,workflow_dataset_combinations
0,<1GB,"[(lcdb.workflow.sklearn.LibLinearWorkflow, 12)..."
1,1GB,"[(lcdb.workflow.sklearn.LibLinearWorkflow, 427..."
